# Energy Prediction - Device-Specific XGBoost (Improved)

This notebook documents the improved XGBoost-based energy prediction pipeline
used in the thesis project.

**What changed from the previous approach:**
- The old notebook (`energy_prediction_model_multidevice.ipynb`) used an
  Extra Trees ensemble as the primary model for both devices.
- This notebook replaces that with a **device-specific XGBoost** strategy:
  - **Jetson Nano**: XGBoost on raw energy target with log-energy stratified
    80/20 split (selected over ExtraTrees when XGBoost achieves higher R²).
  - **Raspberry Pi 5**: XGBoost raw target, which consistently reaches R²≈0.983.
- Both models are trained on the same 26-feature set derived from standard
  model-header statistics (params, GFLOPs, GMACs, latency, throughput).
- Production models are retrained on **all** available data before saving.


In [ ]:
import os
import json
import pickle
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import train_test_split, KFold, cross_val_predict
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")
print("All imports OK")


## 1. Load Data

In [ ]:
JETSON_CSV = "../../data/360_models_benchmark_jetson.csv"
RPI5_CSV   = "../../data/253_models_benchmark_rpi5.csv"

jetson = pd.read_csv(JETSON_CSV)
rpi5   = pd.read_csv(RPI5_CSV)

print(f"Jetson Nano  — shape: {jetson.shape}")
print(jetson["energy_avg_mwh"].describe().rename("energy_avg_mwh (mWh)"))
print()
print(f"Raspberry Pi 5 — shape: {rpi5.shape}")
print(rpi5["energy_avg_mwh"].describe().rename("energy_avg_mwh (mWh)"))


## 2. Feature Engineering (26 features)

In [ ]:
FEATURES = [
    "params_m", "gflops", "gmacs", "size_mb",
    "latency_avg_s", "throughput_iter_per_s",
    "gflops_per_param", "gmacs_per_mb", "latency_throughput_ratio",
    "compute_intensity", "model_complexity", "computational_density",
    "latency_per_gflop", "size_per_param",
    "log_params_m", "log_gflops", "log_size_mb", "log_gmacs",
    "log_latency", "log_throughput", "log_model_complexity", "log_compute_intensity",
    "log_latency_per_gflop",
    "log_params_x_log_latency", "log_gflops_x_log_latency",
    "batch_high_power",
]

print(f"Total features: {len(FEATURES)}")
for i, f in enumerate(FEATURES):
    print(f"  {i+1:2d}. {f}")


def build_features(df, device_type):
    """Add 26 engineered features to the benchmark dataframe.

    Parameters
    ----------
    df : pd.DataFrame
        Raw benchmark CSV loaded into a DataFrame.
    device_type : str
        "jetson_nano" sets batch_high_power=1; any other value sets it to 0.

    Returns
    -------
    pd.DataFrame
        Copy of *df* with all 26 feature columns appended.
    """
    out = df.copy()
    eps = 1e-6

    # Ratio / interaction features
    out["gflops_per_param"]        = out["gflops"] / (out["params_m"] + eps)
    out["gmacs_per_mb"]            = out["gmacs"]  / (out["size_mb"]  + eps)
    out["latency_throughput_ratio"]= out["latency_avg_s"] * out["throughput_iter_per_s"]
    out["compute_intensity"]       = out["gflops"] * out["latency_avg_s"]
    out["model_complexity"]        = out["params_m"] * out["gflops"]
    out["computational_density"]   = out["gflops"] / (out["size_mb"] + eps)
    out["latency_per_gflop"]       = out["latency_avg_s"] / (out["gflops"] + eps)
    out["size_per_param"]          = out["size_mb"] / (out["params_m"] + eps)

    # Log-space features
    out["log_params_m"]            = np.log1p(out["params_m"])
    out["log_gflops"]              = np.log1p(out["gflops"])
    out["log_size_mb"]             = np.log1p(out["size_mb"])
    out["log_gmacs"]               = np.log1p(out["gmacs"])
    out["log_latency"]             = np.log1p(out["latency_avg_s"])
    out["log_throughput"]          = np.log1p(out["throughput_iter_per_s"])
    out["log_model_complexity"]    = np.log1p(out["model_complexity"])
    out["log_compute_intensity"]   = np.log1p(out["compute_intensity"])
    out["log_latency_per_gflop"]   = np.log1p(out["latency_per_gflop"])

    # Cross-log interactions
    out["log_params_x_log_latency"]= out["log_params_m"] * out["log_latency"]
    out["log_gflops_x_log_latency"]= out["log_gflops"]   * out["log_latency"]

    # Device flag: Jetson Nano tends to draw more power at high batch load
    out["batch_high_power"] = 1 if device_type == "jetson_nano" else 0

    out.replace([np.inf, -np.inf], np.nan, inplace=True)
    out.fillna(0, inplace=True)
    return out


## 3. Train Device-Specific XGBoost

In [ ]:
# ── Helper metrics ────────────────────────────────────────────────────────────

def mape(y_true, y_pred):
    mask = np.abs(y_true) > 1e-6
    return float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask]))) * 100

def median_ape(y_true, y_pred):
    mask = np.abs(y_true) > 1e-6
    return float(np.median(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask]))) * 100

def r2_score(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return float(1 - ss_res / (ss_tot + 1e-10))

def pearson_r(y_true, y_pred):
    return float(np.corrcoef(y_true, y_pred)[0, 1])

def get_metrics(y_true, y_pred):
    return {
        "rmse":        float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "mae":         float(mean_absolute_error(y_true, y_pred)),
        "mape":        mape(y_true, y_pred),
        "median_mape": median_ape(y_true, y_pred),
        "r2":          r2_score(y_true, y_pred),
        "pearson_r":   pearson_r(y_true, y_pred),
    }


# ── Jetson Nano ───────────────────────────────────────────────────────────────

def train_jetson(df):
    """Train XGBoost / ExtraTrees on Jetson Nano data.

    Uses stratified 80/20 split by log-energy quantile bins (q=5) so that
    the full energy range is represented in both train and test sets.
    The model with the higher test R² is kept and retrained on ALL data.
    """
    print(f"Jetson Nano ({len(df)} samples)")
    print(f"  Energy: min={df['energy_avg_mwh'].min():.2f}  "
          f"median={df['energy_avg_mwh'].median():.2f}  "
          f"max={df['energy_avg_mwh'].max():.2f} mWh")

    df_feat = build_features(df, "jetson_nano")
    feat_cols = [c for c in FEATURES if c in df_feat.columns]
    X = df_feat[feat_cols].values
    y = df["energy_avg_mwh"].values

    # Stratify by log-energy quantile bins
    bins = pd.qcut(pd.Series(np.log1p(y)), q=5, labels=False, duplicates="drop")
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.20, random_state=42, stratify=bins.values
    )
    print(f"  Train: {len(X_tr)}  Test: {len(X_te)}")
    print(f"  Test energy range: {y_te.min():.2f} – {y_te.max():.2f} mWh  "
          f"mean={y_te.mean():.2f}")

    # ExtraTrees baseline
    et = ExtraTreesRegressor(
        n_estimators=600, max_features=0.65, min_samples_leaf=1,
        max_depth=None, random_state=42, n_jobs=-1
    )
    et.fit(X_tr, y_tr)
    preds_et = np.clip(et.predict(X_te), 0, None)
    m_et = get_metrics(y_te, preds_et)

    # XGBoost challenger
    xgb = XGBRegressor(
        n_estimators=800, learning_rate=0.04, max_depth=7,
        subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.05, reg_lambda=1.5, min_child_weight=2,
        random_state=42, n_jobs=-1, verbosity=0
    )
    xgb.fit(X_tr, y_tr)
    preds_xgb = np.clip(xgb.predict(X_te), 0, None)
    m_xgb = get_metrics(y_te, preds_xgb)

    print(f"  ExtraTrees: RMSE={m_et['rmse']:.3f}  MAE={m_et['mae']:.3f}  "
          f"medMAPE={m_et['median_mape']:.2f}%  R²={m_et['r2']:.4f}")
    print(f"  XGBoost:    RMSE={m_xgb['rmse']:.3f}  MAE={m_xgb['mae']:.3f}  "
          f"medMAPE={m_xgb['median_mape']:.2f}%  R²={m_xgb['r2']:.4f}")

    # Practical range evaluation (<500 mWh = typical deployment budget)
    mask_prac = y_te < 500
    n_prac = int(np.sum(mask_prac))
    m_prac_et  = get_metrics(y_te[mask_prac], preds_et[mask_prac])  if n_prac > 5 else {}
    m_prac_xgb = get_metrics(y_te[mask_prac], preds_xgb[mask_prac]) if n_prac > 5 else {}
    if n_prac > 5:
        print(f"  Practical range (<500 mWh, N={n_prac}/{len(y_te)}):")
        print(f"    ExtraTrees: RMSE={m_prac_et['rmse']:.3f}  "
              f"MAPE={m_prac_et['mape']:.2f}%  R²={m_prac_et['r2']:.4f}")
        print(f"    XGBoost:    RMSE={m_prac_xgb['rmse']:.3f}  "
              f"MAPE={m_prac_xgb['mape']:.2f}%  R²={m_prac_xgb['r2']:.4f}")

    # Pick model with higher R²
    use_xgb  = m_xgb["r2"] > m_et["r2"]
    best_name = "XGBoost" if use_xgb else "ExtraTrees"
    best_m    = m_xgb if use_xgb else m_et
    best_prac = m_prac_xgb if use_xgb else m_prac_et
    preds_best = preds_xgb if use_xgb else preds_et
    print(f"  Selected: {best_name}  (R²={best_m['r2']:.4f})")

    # Production model retrained on ALL data
    if use_xgb:
        prod_model = XGBRegressor(
            n_estimators=800, learning_rate=0.04, max_depth=7,
            subsample=0.8, colsample_bytree=0.8,
            reg_alpha=0.05, reg_lambda=1.5, min_child_weight=2,
            random_state=42, n_jobs=-1, verbosity=0
        )
    else:
        prod_model = ExtraTreesRegressor(
            n_estimators=600, max_features=0.65, min_samples_leaf=1,
            random_state=42, n_jobs=-1
        )
    prod_model.fit(X, y)

    # 5-fold CV for reference
    if use_xgb:
        prod_cv = XGBRegressor(
            n_estimators=800, learning_rate=0.04, max_depth=7,
            subsample=0.8, colsample_bytree=0.8,
            reg_alpha=0.05, reg_lambda=1.5, min_child_weight=2,
            random_state=42, n_jobs=-1, verbosity=0
        )
    else:
        prod_cv = ExtraTreesRegressor(
            n_estimators=600, max_features=0.65, random_state=42, n_jobs=-1
        )
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_preds = np.clip(cross_val_predict(prod_cv, X, y, cv=kf), 0, None)
    cv_m = get_metrics(y, cv_preds)
    print(f"  5-fold CV: RMSE={cv_m['rmse']:.3f}  "
          f"medMAPE={cv_m['median_mape']:.2f}%  R²={cv_m['r2']:.4f}")

    return prod_model, feat_cols, best_m, best_prac, cv_m, best_name, y_te, preds_best


# ── Raspberry Pi 5 ────────────────────────────────────────────────────────────

def train_rpi5(df):
    """Train XGBoost on Raspberry Pi 5 data.

    XGBoost on raw energy target consistently reaches R²≈0.983 for RPi5.
    """
    print(f"\nRaspberry Pi 5 ({len(df)} samples)")
    print(f"  Energy: min={df['energy_avg_mwh'].min():.2f}  "
          f"median={df['energy_avg_mwh'].median():.2f}  "
          f"max={df['energy_avg_mwh'].max():.2f} mWh")

    df_feat = build_features(df, "raspberry_pi5")
    feat_cols = [c for c in FEATURES if c in df_feat.columns]
    X = df_feat[feat_cols].values
    y = df["energy_avg_mwh"].values

    bins = pd.qcut(pd.Series(np.log1p(y)), q=5, labels=False, duplicates="drop")
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.20, random_state=42, stratify=bins.values
    )
    print(f"  Train: {len(X_tr)}  Test: {len(X_te)}")
    print(f"  Test energy range: {y_te.min():.2f} – {y_te.max():.2f} mWh")

    xgb = XGBRegressor(
        n_estimators=800, learning_rate=0.04, max_depth=6,
        subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.05, reg_lambda=1.0, min_child_weight=2,
        random_state=42, n_jobs=-1, verbosity=0
    )
    xgb.fit(X_tr, y_tr)
    preds = np.clip(xgb.predict(X_te), 0, None)
    m = get_metrics(y_te, preds)
    print(f"  XGBoost: RMSE={m['rmse']:.3f}  MAE={m['mae']:.3f}  "
          f"MAPE={m['mape']:.2f}%  medMAPE={m['median_mape']:.2f}%  "
          f"R²={m['r2']:.4f}  Pearson r={m['pearson_r']:.4f}")

    # Production model on ALL data
    prod_model = XGBRegressor(
        n_estimators=800, learning_rate=0.04, max_depth=6,
        subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.05, reg_lambda=1.0, min_child_weight=2,
        random_state=42, n_jobs=-1, verbosity=0
    )
    prod_model.fit(X, y)

    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    prod_cv = XGBRegressor(
        n_estimators=800, learning_rate=0.04, max_depth=6,
        subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.05, reg_lambda=1.0, min_child_weight=2,
        random_state=42, n_jobs=-1, verbosity=0
    )
    cv_preds = np.clip(cross_val_predict(prod_cv, X, y, cv=kf), 0, None)
    cv_m = get_metrics(y, cv_preds)
    print(f"  5-fold CV: RMSE={cv_m['rmse']:.3f}  "
          f"MAPE={cv_m['mape']:.2f}%  R²={cv_m['r2']:.4f}")

    return prod_model, feat_cols, m, cv_m, y_te, preds


# ── Run training ──────────────────────────────────────────────────────────────
print("=" * 65)
(j_model, j_feats, j_test, j_prac, j_cv, j_name,
 j_y_te, j_preds) = train_jetson(jetson)

(r_model, r_feats, r_test, r_cv,
 r_y_te, r_preds) = train_rpi5(rpi5)


## 4. Results

In [ ]:
# ── Combined test metrics ─────────────────────────────────────────────────────
n_j = round(len(jetson) * 0.2)
n_r = round(len(rpi5)   * 0.2)
n_total = n_j + n_r

comb_rmse    = float(np.sqrt((n_j * j_test["rmse"]**2 + n_r * r_test["rmse"]**2) / n_total))
comb_mae     = float((n_j * j_test["mae"]         + n_r * r_test["mae"])         / n_total)
comb_mape    = float((n_j * j_test["mape"]        + n_r * r_test["mape"])        / n_total)
comb_medmape = float((n_j * j_test["median_mape"] + n_r * r_test["median_mape"]) / n_total)
comb_r2      = float((n_j * j_test["r2"]          + n_r * r_test["r2"])          / n_total)
comb_pearson = float((n_j * j_test["pearson_r"]   + n_r * r_test["pearson_r"])   / n_total)

# Expected targets from train_final.py (for verification):
#   Jetson  — RMSE=177.335  medMAPE=14.28%  R²=0.9359
#   RPi5    — RMSE=5.628    MAPE=9.66%      R²=0.9826
#   Combined— RMSE=135.726  medMAPE=12.04%  R²=0.9553  Pearson r=0.9826

print("=" * 65)
print(f"FINAL RESULTS  ({n_total} test samples: {n_j} Jetson + {n_r} RPi5)")
print("=" * 65)
header = f"{'Metric':<20} {'Jetson Nano':>14} {'RPi5':>12} {'Combined':>12}"
print(header)
print("-" * len(header))
print(f"{'RMSE (mWh)':<20} {j_test['rmse']:>14.3f} {r_test['rmse']:>12.3f} {comb_rmse:>12.3f}")
print(f"{'MAE (mWh)':<20} {j_test['mae']:>14.3f} {r_test['mae']:>12.3f} {comb_mae:>12.3f}")
print(f"{'Mean MAPE (%)':<20} {j_test['mape']:>14.2f} {r_test['mape']:>12.2f} {comb_mape:>12.2f}")
print(f"{'Median MAPE (%)':<20} {j_test['median_mape']:>14.2f} {r_test['median_mape']:>12.2f} {comb_medmape:>12.2f}")
print(f"{'R²':<20} {j_test['r2']:>14.4f} {r_test['r2']:>12.4f} {comb_r2:>12.4f}")
print(f"{'Pearson r':<20} {j_test['pearson_r']:>14.4f} {r_test['pearson_r']:>12.4f} {comb_pearson:>12.4f}")
print("=" * 65)
print(f"Jetson model: {j_name}")
if j_prac:
    print(f"Jetson practical (<500 mWh): "
          f"RMSE={j_prac['rmse']:.3f}  MAPE={j_prac['mape']:.2f}%  R²={j_prac['r2']:.4f}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Jetson panel
ax = axes[0]
ax.scatter(j_y_te, j_preds, alpha=0.6, edgecolors="none", s=25)
lo = min(j_y_te.min(), j_preds.min())
hi = max(j_y_te.max(), j_preds.max())
ax.plot([lo, hi], [lo, hi], "r--", linewidth=1.2, label="Perfect fit")
ax.set_xlabel("Actual energy (mWh)")
ax.set_ylabel("Predicted energy (mWh)")
ax.set_title(f"Jetson Nano — {j_name}\nR²={j_test['r2']:.4f}  medMAPE={j_test['median_mape']:.2f}%")
ax.legend()

# RPi5 panel
ax = axes[1]
ax.scatter(r_y_te, r_preds, alpha=0.6, edgecolors="none", s=25, color="darkorange")
lo = min(r_y_te.min(), r_preds.min())
hi = max(r_y_te.max(), r_preds.max())
ax.plot([lo, hi], [lo, hi], "r--", linewidth=1.2, label="Perfect fit")
ax.set_xlabel("Actual energy (mWh)")
ax.set_ylabel("Predicted energy (mWh)")
ax.set_title(f"Raspberry Pi 5 — XGBoost\nR²={r_test['r2']:.4f}  MAPE={r_test['mape']:.2f}%")
ax.legend()

plt.tight_layout()
plt.savefig("actual_vs_predicted.png", dpi=120)
plt.show()
print("Plot saved to actual_vs_predicted.png")


## 5. Save Models

In [ ]:
ARTIFACTS_DIR = "../../artifacts"
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

# Save production models
with open(os.path.join(ARTIFACTS_DIR, "jetson_energy_model.pkl"), "wb") as f:
    pickle.dump(j_model, f)
print(f"Saved jetson_energy_model.pkl  ({j_name})")

with open(os.path.join(ARTIFACTS_DIR, "rpi5_energy_model.pkl"), "wb") as f:
    pickle.dump(r_model, f)
print("Saved rpi5_energy_model.pkl  (XGBoost)")

# Scalers (fitted on full dataset — used by energy_predictor_service.py)
scaler_j = StandardScaler()
scaler_r = StandardScaler()
dj = build_features(jetson, "jetson_nano")
dr = build_features(rpi5,   "raspberry_pi5")
Xj = dj[[c for c in FEATURES if c in dj.columns]].values
Xr = dr[[c for c in FEATURES if c in dr.columns]].values
scaler_j.fit(Xj)
scaler_r.fit(Xr)

with open(os.path.join(ARTIFACTS_DIR, "jetson_scaler.pkl"), "wb") as f:
    pickle.dump(scaler_j, f)
with open(os.path.join(ARTIFACTS_DIR, "rpi5_scaler.pkl"), "wb") as f:
    pickle.dump(scaler_r, f)
print("Saved jetson_scaler.pkl and rpi5_scaler.pkl")

# Feature list
with open(os.path.join(ARTIFACTS_DIR, "device_specific_features.json"), "w") as f:
    json.dump(FEATURES, f, indent=2)
print("Saved device_specific_features.json")

# Metadata
metadata = {
    "jetson_model": {
        "model_name": j_name,
        "log_transform_target": False,
        "metrics": {
            "test_rmse":        j_test["rmse"],
            "test_mae":         j_test["mae"],
            "test_mape":        j_test["mape"],
            "test_median_mape": j_test["median_mape"],
            "test_r2":          j_test["r2"],
            "test_pearson_r":   j_test["pearson_r"],
            "practical_rmse":   j_prac.get("rmse"),
            "practical_mape":   j_prac.get("mape"),
            "practical_r2":     j_prac.get("r2"),
            "cv_rmse":          j_cv["rmse"],
            "cv_mape":          j_cv["mape"],
            "cv_median_mape":   j_cv["median_mape"],
            "cv_r2":            j_cv["r2"],
        },
        "feature_count": len(j_feats),
    },
    "rpi5_model": {
        "model_name": "XGBoost",
        "log_transform_target": False,
        "metrics": {
            "test_rmse":        r_test["rmse"],
            "test_mae":         r_test["mae"],
            "test_mape":        r_test["mape"],
            "test_median_mape": r_test["median_mape"],
            "test_r2":          r_test["r2"],
            "test_pearson_r":   r_test["pearson_r"],
            "cv_rmse":          r_cv["rmse"],
            "cv_mape":          r_cv["mape"],
            "cv_r2":            r_cv["r2"],
        },
        "feature_count": len(r_feats),
    },
    "combined_test": {
        "n_total":     int(n_total),
        "n_jetson":    int(n_j),
        "n_rpi5":      int(n_r),
        "rmse":        comb_rmse,
        "mae":         comb_mae,
        "mape":        comb_mape,
        "median_mape": comb_medmape,
        "r2":          comb_r2,
        "pearson_r":   comb_pearson,
    },
}
with open(os.path.join(ARTIFACTS_DIR, "device_specific_metadata.json"), "w") as f:
    json.dump(metadata, f, indent=2)
print("Saved device_specific_metadata.json")

# Energy thresholds (percentile-based, used by energy_predictor_service.py)
j_e   = jetson["energy_avg_mwh"].dropna()
r_e   = rpi5["energy_avg_mwh"].dropna()
all_e = pd.concat([j_e, r_e])
thresholds = {
    "jetson_nano": {
        "p25": float(np.percentile(j_e, 25)),
        "p50": float(np.percentile(j_e, 50)),
        "p75": float(np.percentile(j_e, 75)),
    },
    "raspberry_pi5": {
        "p25": float(np.percentile(r_e, 25)),
        "p50": float(np.percentile(r_e, 50)),
        "p75": float(np.percentile(r_e, 75)),
    },
    "unified": {
        "p25": float(np.percentile(all_e, 25)),
        "p50": float(np.percentile(all_e, 50)),
        "p75": float(np.percentile(all_e, 75)),
    },
}
with open(os.path.join(ARTIFACTS_DIR, "energy_thresholds.json"), "w") as f:
    json.dump(thresholds, f, indent=2)
print("Saved energy_thresholds.json")

print("\n[OK] All artifacts saved to", os.path.abspath(ARTIFACTS_DIR))
